# Circuit Evaluation

Discovering a circuit is only half the job — you need to evaluate whether
it's actually a good explanation. This notebook covers the three standard
circuit evaluation metrics: **faithfulness**, **completeness**, and
**minimality**.

This notebook covers the `irtk.circuit_evaluation` module.

## Why This Matters

Once you find a circuit, how good is it? Circuit evaluation measures faithfulness (does the circuit reproduce the full model's behavior?), completeness (does it capture all relevant computation?), and minimality (are there unnecessary components?). These metrics are essential for rigorous circuit claims.

**Key references:**
- [Wang et al. (2023) "Interpretability in the Wild: IOI"](https://arxiv.org/abs/2211.00593) — Detailed circuit analysis of indirect object identification
- [Conmy et al. (2023) "Towards Automated Circuit Discovery"](https://arxiv.org/abs/2304.14997) — Automated methods for circuit finding via edge pruning

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk import circuit_evaluation, experiments
from irtk.patching import make_logit_diff_metric

model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Model: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads")

## 1. Discover a Circuit

First, let's find a circuit for a simple task using `experiments.find_circuit`.

In [ ]:
# Simple factual recall task
prompt = "The Eiffel Tower is located in"
tokens = model.to_tokens(prompt)
token_strs = [tokenizer.decode([int(t)]) for t in tokens]

# Check the prediction
logits = model(tokens)
top_pred = int(jnp.argmax(logits[-1]))
print(f"Prompt: {prompt!r}")
print(f"Top prediction: {tokenizer.decode([top_pred])!r}")

# Metric: logit for " Paris"
paris_id = tokenizer.encode(" Paris")[0]
def metric(logits):
    return float(logits[-1, paris_id])

# Discover circuit
circuit_result = experiments.find_circuit(
    model, [tokens], metric, threshold=0.05, method="zero"
)

circuit_heads = [(l, h) for l, h, _ in circuit_result.values["circuit_heads"]]
print(f"\nDiscovered circuit: {len(circuit_heads)} heads")
for l, h, imp in circuit_result.values["circuit_heads"]:
    print(f"  L{l}H{h}: importance={imp:.4f}")

## 2. Faithfulness Score

Does the circuit reproduce the full model's behavior?
We ablate everything OUTSIDE the circuit and check how much of the
metric is preserved.

In [ ]:
faith = circuit_evaluation.faithfulness_score(
    model, circuit_heads, [tokens], metric
)
print(f"Faithfulness: {faith:.4f}")
print(f"  1.0 = circuit perfectly reproduces full model")
print(f"  0.0 = circuit has no predictive power")

## 3. Completeness Score

Is the circuit sufficient? We ablate everything INSIDE the circuit and
check how much the metric drops.

In [ ]:
comp = circuit_evaluation.completeness_score(
    model, circuit_heads, [tokens], metric
)
print(f"Completeness: {comp:.4f}")
print(f"  1.0 = circuit contains ALL the behavior")
print(f"  0.0 = circuit is irrelevant to the behavior")

## 4. Minimality Check

Can any component be removed without significant loss? This identifies
redundant heads in the circuit.

In [ ]:
mini = circuit_evaluation.minimality_check(
    model, circuit_heads, [tokens], metric, threshold=0.05
)

print(f"Minimality analysis ({len(mini)} heads):")
for entry in mini:
    status = "REDUNDANT" if entry["redundant"] else "necessary"
    print(f"  L{entry['layer']}H{entry['head']}: "
          f"change={entry['relative_change']:.4f} [{status}]")

n_redundant = sum(1 for e in mini if e["redundant"])
print(f"\n{n_redundant}/{len(mini)} heads are redundant")

## 5. Full Evaluation

Run all metrics at once with `evaluate_circuit`.

In [ ]:
result = circuit_evaluation.evaluate_circuit(
    model, circuit_heads, [tokens], metric
)

print("=== Circuit Evaluation Report ===")
print(f"Circuit size: {result['n_circuit_heads']} heads")
print(f"Baseline metric: {result['baseline_metric']:.4f}")
print(f"Circuit metric:  {result['circuit_metric']:.4f}")
print(f"")
print(f"Faithfulness:  {result['faithfulness']:.4f}")
print(f"Completeness:  {result['completeness']:.4f}")
print(f"Redundant heads: {result['n_redundant']}/{result['n_circuit_heads']}")

## 6. Circuit IoU

Compare two circuits to see how much they overlap.

In [ ]:
# Discover circuit with different threshold
circuit_result_strict = experiments.find_circuit(
    model, [tokens], metric, threshold=0.1, method="zero"
)
strict_heads = [(l, h) for l, h, _ in circuit_result_strict.values["circuit_heads"]]

iou = circuit_evaluation.circuit_iou(circuit_heads, strict_heads)
print(f"Loose circuit: {len(circuit_heads)} heads")
print(f"Strict circuit: {len(strict_heads)} heads")
print(f"IoU: {iou:.4f}")

# Visualize overlap
set_loose = set(circuit_heads)
set_strict = set(strict_heads)
only_loose = set_loose - set_strict
both = set_loose & set_strict
only_strict = set_strict - set_loose
print(f"\nOnly in loose: {sorted(only_loose)}")
print(f"In both: {sorted(both)}")
print(f"Only in strict: {sorted(only_strict)}")

In [ ]:
# Visualize circuit on model grid
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, heads, title in [
    (axes[0], circuit_heads, f"Loose (threshold=0.05, {len(circuit_heads)} heads)"),
    (axes[1], strict_heads, f"Strict (threshold=0.10, {len(strict_heads)} heads)"),
]:
    grid = np.zeros((model.cfg.n_layers, model.cfg.n_heads))
    for l, h in heads:
        grid[l, h] = 1.0
    ax.imshow(grid, cmap="Blues", aspect="auto", vmin=0, vmax=1)
    ax.set_xlabel("Head")
    ax.set_ylabel("Layer")
    ax.set_title(title)

plt.suptitle(f"Circuit Comparison (IoU={iou:.2f})", fontsize=14)
plt.tight_layout()
plt.show()

## Summary

| Function | Purpose |
|----------|--------|
| `faithfulness_score()` | Does the circuit reproduce the full model's behavior? |
| `completeness_score()` | Does ablating the circuit destroy the behavior? |
| `minimality_check()` | Can any component be removed without loss? |
| `circuit_iou()` | Intersection-over-union between two circuits |
| `evaluate_circuit()` | All metrics in one call |